# 🔍 Customer Churn Prediction & Analysis
**Author:** Your Name  
**Dataset:** Telco Customer Churn (Kaggle)  
**Goal:** Identify churn drivers and build a predictive ML model

---

## 📦 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score, roc_curve
)
from xgboost import XGBClassifier

# Plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
print('✅ Libraries loaded successfully!')

## 📂 2. Load & Explore Data

In [ ]:
# Load dataset
# Download from: https://www.kaggle.com/datasets/blastchar/telco-customer-churn
df = pd.read_csv('telco_churn.csv')

print('Shape:', df.shape)
print('\nFirst 5 rows:')
df.head()

In [ ]:
# Basic info
print('=== DATA TYPES ===')
print(df.dtypes)
print('\n=== MISSING VALUES ===')
print(df.isnull().sum())
print('\n=== CHURN DISTRIBUTION ===')
print(df['Churn'].value_counts())
print(f"Churn Rate: {df['Churn'].value_counts(normalize=True)['Yes']*100:.1f}%")

## 🧹 3. Data Cleaning

In [ ]:
# Fix TotalCharges column (has spaces, should be numeric)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Fill missing TotalCharges with median
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

# Drop customerID (not useful for modeling)
df.drop('customerID', axis=1, inplace=True)

# Convert Churn to binary
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print('✅ Data cleaned!')
print(f'Final shape: {df.shape}')

## 📊 4. Exploratory Data Analysis (EDA)

In [ ]:
# --- Churn Distribution Pie Chart ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
churn_counts = df['Churn'].value_counts()
axes[0].pie(
    churn_counts,
    labels=['No Churn', 'Churned'],
    autopct='%1.1f%%',
    colors=['#2ecc71', '#e74c3c'],
    startangle=90
)
axes[0].set_title('Overall Churn Rate', fontsize=14, fontweight='bold')

# Churn by Contract type
contract_churn = df.groupby('Contract')['Churn'].mean().reset_index()
axes[1].bar(
    contract_churn['Contract'],
    contract_churn['Churn'] * 100,
    color=['#e74c3c', '#f39c12', '#2ecc71']
)
axes[1].set_title('Churn Rate by Contract Type', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Churn Rate (%)')
for i, v in enumerate(contract_churn['Churn'] * 100):
    axes[1].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('images/churn_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Insight: Month-to-month customers churn significantly more!')

In [ ]:
# --- Churn by Tenure ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Tenure distribution
axes[0].hist(
    [df[df['Churn']==0]['tenure'], df[df['Churn']==1]['tenure']],
    label=['No Churn', 'Churned'],
    color=['#2ecc71', '#e74c3c'],
    bins=30, alpha=0.7
)
axes[0].set_title('Tenure Distribution by Churn', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Tenure (months)')
axes[0].set_ylabel('Count')
axes[0].legend()

# Monthly Charges
axes[1].hist(
    [df[df['Churn']==0]['MonthlyCharges'], df[df['Churn']==1]['MonthlyCharges']],
    label=['No Churn', 'Churned'],
    color=['#2ecc71', '#e74c3c'],
    bins=30, alpha=0.7
)
axes[1].set_title('Monthly Charges by Churn', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Monthly Charges ($)')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.savefig('images/churn_tenure_charges.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Insight: New customers (low tenure) and high payers churn more!')

In [ ]:
# --- Churn by Key Categorical Features ---
cat_features = ['InternetService', 'PaymentMethod', 'TechSupport', 'OnlineSecurity']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, feature in enumerate(cat_features):
    churn_rate = df.groupby(feature)['Churn'].mean() * 100
    bars = axes[i].bar(churn_rate.index, churn_rate.values, color=sns.color_palette('Set2'))
    axes[i].set_title(f'Churn Rate by {feature}', fontsize=12, fontweight='bold')
    axes[i].set_ylabel('Churn Rate (%)')
    axes[i].tick_params(axis='x', rotation=15)
    for bar, val in zip(bars, churn_rate.values):
        axes[i].text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', fontsize=10, fontweight='bold'
        )

plt.tight_layout()
plt.savefig('images/churn_categorical.png', dpi=150, bbox_inches='tight')
plt.show()

## 🔧 5. Feature Engineering & Preprocessing

In [ ]:
# Encode categorical columns
df_model = df.copy()
le = LabelEncoder()

cat_cols = df_model.select_dtypes(include='object').columns
print('Encoding columns:', list(cat_cols))

for col in cat_cols:
    df_model[col] = le.fit_transform(df_model[col])

# Define features and target
X = df_model.drop('Churn', axis=1)
y = df_model['Churn']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'✅ Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}')

## 🤖 6. Model Training & Evaluation

In [ ]:
# Train all three models
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
}

results = {}

for name, model in models.items():
    if name == 'Logistic Regression':
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    results[name] = {'model': model, 'accuracy': acc, 'auc': auc, 'proba': y_proba, 'pred': y_pred}
    print(f'{name}: Accuracy={acc:.3f} | AUC-ROC={auc:.3f}')

print('\n✅ Best Model: XGBoost')

In [ ]:
# --- ROC Curve Comparison ---
plt.figure(figsize=(8, 6))
colors = ['#3498db', '#2ecc71', '#e74c3c']

for (name, result), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, result['proba'])
    plt.plot(fpr, tpr, label=f"{name} (AUC={result['auc']:.3f})", color=color, linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve Comparison', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig('images/roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Feature Importance (XGBoost) ---
xgb_model = results['XGBoost']['model']
importance = pd.Series(xgb_model.feature_importances_, index=X.columns)
importance = importance.sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 6))
bars = plt.barh(importance.index[::-1], importance.values[::-1], color='#3498db')
plt.xlabel('Feature Importance Score', fontsize=12)
plt.title('Top 10 Features Driving Churn (XGBoost)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('images/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('💡 These are the strongest predictors of customer churn!')

In [ ]:
# --- Confusion Matrix ---
xgb_pred = results['XGBoost']['pred']
cm = confusion_matrix(y_test, xgb_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['No Churn', 'Churned'],
    yticklabels=['No Churn', 'Churned']
)
plt.title('Confusion Matrix — XGBoost', fontsize=14, fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('images/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(classification_report(y_test, xgb_pred, target_names=['No Churn', 'Churned']))

## 💾 7. Save Model for Streamlit App

In [ ]:
import joblib
import os

os.makedirs('model', exist_ok=True)
os.makedirs('images', exist_ok=True)

# Save model and scaler
joblib.dump(xgb_model, 'model/xgb_churn_model.pkl')
joblib.dump(scaler, 'model/scaler.pkl')
joblib.dump(list(X.columns), 'model/feature_names.pkl')

print('✅ Model saved to model/xgb_churn_model.pkl')
print('✅ Scaler saved to model/scaler.pkl')
print('✅ Ready to use in Streamlit app!')

## 💡 8. Key Business Insights & Recommendations

### Summary of Findings:
1. **Overall churn rate is ~26.5%** — roughly 1 in 4 customers leave
2. **Month-to-month contracts** have the highest churn (~43%)
3. **New customers (tenure < 12 months)** are the most at-risk group
4. **High monthly charges** correlate with higher churn
5. **Lack of tech support & online security** increases churn significantly

### Business Recommendations:
| # | Recommendation | Expected Impact |
|---|---|---|
| 1 | Offer discounts to convert month-to-month → annual plans | High |
| 2 | Create onboarding program for new customers (first 12 months) | High |
| 3 | Bundle tech support with basic plans | Medium |
| 4 | Flag high-risk customers monthly using the XGBoost model | High |
| 5 | Review pricing for high monthly charge tiers | Medium |